# DRC CoRL 2026 — RUN ALL (full reduced-scope experiment, end-to-end)

One consolidated notebook that runs the whole thing on Kaggle: sim install → Part B (real LIBERO) → Part A (controlled theory validation) → analysis + figures.

**Scope (reduced for TMLR/CoRL):** 3 LIBERO tasks spanning the horizon axis — `LIBERO-Spatial-1` (short), `LIBERO-Goal-1` (medium), `LIBERO-Long-1` (long) — × **both** architectures (`diffusion`, `act`) × **3 seeds**. Robomimic has been dropped.

**Accelerator:** set Settings → Accelerator → **GPU T4 x2** (the training script pins one job per GPU). A single T4 also works — run Part B with `SEEDS="0 1"`.

**Order of cells:**
1. Clone / pull the repo.
2. Confirm GPUs are visible.
3. Install the sim stack (`kaggle_setup.sh`) — run alone first.
4. Validate the install (builds + renders one real LIBERO env).
5. Record the pre-registration hashes (locked `analysis.py` / `config.py`).
6. **Part B** — the sim experiment (train → metrics → rollouts).
7. **Part A** — controlled theory validation (CPU, fast).
8. **Analysis + figures** (H1–H4).

Each stage writes to `results/` and `figures/`; the analysis stage reads those back.

## 1. Clone or pull the repo

In [ ]:
REPO_URL = "https://github.com/keshavkrishnan08/CORL.git"
BRANCH   = "main"
%cd /kaggle/working
![ -d CORL ] && git -C CORL pull || git clone -b $BRANCH $REPO_URL CORL
%cd /kaggle/working/CORL
!git log --oneline -1

## 2. Confirm GPUs are visible

Expect **2** for T4 x2. A single process only uses `cuda:0`; the training script launches one process per GPU. If this says 1 and you selected T4 x2, the second GPU did not activate.

In [ ]:
import torch
print('GPUs visible to torch:', torch.cuda.device_count(), '<-- expect 2 for T4 x2')
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2'

## 3. Install the simulation stack

Installs LIBERO from source plus the pinned robosuite 1.4.0 / mujoco 3.1.6 / numpy<2 deps (see `scripts/kaggle_setup.sh`; not `pip -r`). **Run this cell alone first** and let it finish its import check before going further. Takes a few minutes.

In [ ]:
!bash scripts/kaggle_setup.sh

## 4. Validate the install (builds + renders one real LIBERO env)

**STOP and fix if this fails** — do not spend a training session on a broken install. `check_install.py` sets `MUJOCO_GL=egl` itself; we read from `/dev/null` so LIBERO never blocks on an interactive prompt.

In [ ]:
!python scripts/check_install.py < /dev/null

## 5. Record the pre-registration hashes

`drc/analysis.py` and `drc/config.py` are locked + SHA-256-hashed before training. Print them here so the run is traceable to the pre-registered code.

In [ ]:
from drc.utils import sha256_file
print('analysis.py', sha256_file('drc/analysis.py'))
print('config.py  ', sha256_file('drc/config.py'))

## 6. Part B — the sim experiment (train → metrics → rollouts)

Trains the **3 LIBERO tasks × 2 architectures × 3 seeds**, then computes the **8 offline metrics** and **20 rollouts per checkpoint**, pruning checkpoints per task to stay under the disk cap. Data is downloaded per task (download → train → eval → delete), so peak disk stays a few GB.

**Resumable:** completed tasks are saved and their checkpoints pruned, so if a 12h session times out, just re-run this cell — it picks up the remaining tasks. The script exports `MUJOCO_GL=egl` itself.

Takes **hours**. For a single T4, use the commented line with `SEEDS="0 1"`.

In [ ]:
!bash scripts/run_session1.sh
# single T4:  !SEEDS="0 1" bash scripts/run_session1.sh

In [ ]:
# checkpoint: LIBERO rows are now in results/metrics.csv + results/rollouts.csv
import pandas as pd
print(pd.read_csv('results/rollouts.csv').groupby('task')['success_rate'].mean())

## 7. Part A — controlled theory validation (CPU, fast)

The "cool" numerical sim. On a synthetic system that *meets the theory's assumptions*, it shows: the compounding bound is tight (P1, $R^2\approx1$), validation loss can't rank varying-gain policies while an environment-querying replay can (P2), selection regret (rollout-free ≈ random, env-querying = oracle), and the **horizon wall** (Cor. 1).

This is a *controlled* validation of the theory's mechanism — **not** a real-robot result (that's Part B above). Runs on CPU in seconds to a couple of minutes.

Both scripts take no required arguments (their `main()` uses fixed constants / argparse defaults):
- `run_partA_bound.py` — P1/P2/selection-regret/horizon-wall → `results/partA_bound.json`, `figures/partA/figA1..4`.
- `run_partA_sweep.py` — controlled L-sweep → `results/sweep_summary.csv`, `figures/partA/fig5,fig6`.

In [ ]:
!python scripts/run_partA_bound.py
!python scripts/run_partA_sweep.py

## 8. Analysis + figures (H1–H4)

Runs the locked analysis on `results/metrics.csv` + `results/rollouts.csv`: tests H1–H4, runs the power report, generates all figures, and (if `results/lyapunov.json` exists from Part B's rollouts) the real-data gap-vs-amplification and metric-class-vs-$\hat L$ figures. Writes `results/results.json` and prints the H1–H4 support flags + outcome-matrix row.

`05_analysis.py` defaults `--metrics`/`--rollouts`/`--out` to the standard `results/` paths, so no args are needed.

In [ ]:
!python scripts/05_analysis.py

## Where the outputs land

- `results/metrics.csv`, `results/rollouts.csv` — per-checkpoint offline metrics + rollout success (Part B).
- `results/results.json` — H1–H4 outcomes, power, outcome-matrix row (the analysis stage).
- `results/partA_bound.json`, `results/sweep_summary.csv` — Part A controlled-validation numbers.
- `results/lyapunov.json` — estimated $\hat L$ per task (written by `04_rollouts.py` during Part B); enables the real-data headline figures.
- `figures/` — all paper figures, including `figures/partA/`.

Feed the numbers into the paper macros with `python scripts/fill_paper.py` (reads `results/results.json`).

**Never use the `drc.devtools` fabricated tables (`make_fake_results`) in the paper** — they exist only for code-path verification.